# Step 1 · Historical cycling networks with a Level of Traffic Stress (LTS)

**Question:** how did the cycling network of each city evolve between 2017 and 2026, and how *comfortable* is it?

For every city and every year this notebook

1. downloads the street network from **OpenStreetMap** (OSM) — for 2017–2022 the map *as it was on 31 December*
   of that year (OSM keeps its full history), for 2026 the current map;
2. gives every street segment a **Level of Traffic Stress** for cyclists, from LTS 1 (comfortable for children)
   to LTS 4 (only for confident cyclists), using its OSM tags (type of cycle facility, speed limit, number of lanes);
3. for 2026 only, adds the car roads that have no cycling facility as **LTS 5** — a *planning layer* of streets
   that could be upgraded later (never counted as cycling network);
4. **simplifies** the network (one undirected edge per street, nodes kept only at junctions and where the LTS
   changes; lengths are preserved);
5. computes two **infrastructure metrics** per year:
   * **normalized network length** `N` = cycling km / largest yearly cycling km of the city;
   * **weighted largest connected component** `C` = how far you can ride without exceeding LTS 1, 2, 3, 4,
     combined with weights 0.64 / 0.24 / 0.09 / 0.03.

| | |
|---|---|
| **Input** | `data/city_polygons.geojson` (city boundaries) + `data/osm_snapshots/<city>.tar.xz` (the OpenStreetMap data of the published run, in git) — internet only for a new run |
| **Output** | `results/lts_networks/<city>/<run>/<year>/<city>_lts_<year>_SIMPLE.pkl` (networks) and `<city>_2017_2022_2026_SIMPLIFIED_metrics.csv/.txt` |
| **Run time** | seconds if the networks already exist; 2–10 min per city rebuilt from the shipped data; 10–60 min per city for a new download |

The code lives in [`cyclelts/lts.py`](../cyclelts/lts.py) (classifier), [`cyclelts/network.py`](../cyclelts/network.py)
(download and build), [`cyclelts/simplify.py`](../cyclelts/simplify.py) and [`cyclelts/metrics.py`](../cyclelts/metrics.py).

> **No internet needed.** For the published run (`fresh_2026-10-21`) the raw OpenStreetMap answers of all 11 cities
> are shipped in `data/osm_snapshots/`; they are unpacked and used automatically, so any city is rebuilt exactly as
> published. The finished Grenoble networks are also shipped (loaded instantly). A **new** `RUN_NAME` downloads
> fresh data from the public Overpass servers instead (one city at a time: they refuse parallel downloads).

In [1]:
# Setup: find the repository root and make the `cyclelts` package importable.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cyclelts").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## 1. Settings

In [2]:
import networkx as nx
import pandas as pd
from IPython.display import display

from cyclelts import config
from cyclelts.lts import configure_osmnx, normalize_polygon
from cyclelts.network import CityNetworkBuilder
from cyclelts.metrics import annual_metrics, exponential_weights, write_metrics_files
from cyclelts.simplify import check_graph
from cyclelts.plotting import plot_topology, plot_before_after, leftover_degree2

# ---- CHANGE THESE ---------------------------------------------------------------------
CITIES_TO_BUILD = ["Grenoble"]      # any of config.CITIES, e.g. ["Angers", "Tours"]
RUN_NAME = config.DEFAULT_RUN       # published run: rebuilt offline from data/osm_snapshots/
                                    # a NEW name = completely fresh download into a new folder (needs internet);
                                    # the same name resumes a run (finished years are not built again)
REBUILD = False                     # True = ignore the pickles of THIS run folder and redo every year
MATCH = "way"                       # rule for adding 2026 car roads: "way" (recommended) or "endpoints"
KAPPA = 1.0                         # decay of the LTS weights in the weighted LCC
TIMEOUT = 600                       # seconds the Overpass server may work on one query
SHOW_MAPS = True                    # draw the networks (interactive Plotly maps)
# For a commune that is not in the study, also pass communes_file=<path to communes-20220101.json>
# to config.load_city_polygon (see data/README.md).
# ---------------------------------------------------------------------------------------

configure_osmnx()   # keep the cycleway / maxspeed / lanes / bicycle tags needed by the classifier
print("Studied cities:", [c.name for c in config.CITIES])
print("Years:", config.ALL_YEARS)

Studied cities: ['Angers', 'Annecy', 'Bordeaux', 'Chambéry', 'Grenoble', 'Lyon', 'Nantes', 'Paris', 'Rennes', 'Strasbourg', 'Tours']
Years: [2017, 2018, 2019, 2020, 2021, 2022, 2026]


## 2. Build (or load) the seven networks of each city

Each year is saved as soon as it is finished. If a download fails (busy server), simply run the cell again:
finished years are loaded from disk and only the missing ones are downloaded.

In [3]:
networks, failed = {}, {}
for name in CITIES_TO_BUILD:
    city = config.get_city(name)
    polygon = normalize_polygon(config.load_city_polygon(city.name, city.insee))
    builder = CityNetworkBuilder(city.name, city.slug, polygon, config.run_dir(city.name, RUN_NAME),
                                 rebuild=REBUILD, timeout=TIMEOUT, match=MATCH)
    networks[city.name] = {}
    for year in config.ALL_YEARS:
        try:
            networks[city.name][year] = builder.build_year(year)
        except Exception as exc:          # keep the other years going; rerun the cell later
            failed[(city.name, year)] = repr(exc)
            print(f"[{city.name} {year}] FAILED: {exc}")
print("Failed:", failed or "none")

Unpacked the shipped OpenStreetMap data data\osm_snapshots\grenoble.tar.xz -> results\lts_networks\grenoble\fresh_2026-10-21\osm_snapshot
[2017] loaded results\lts_networks\grenoble\fresh_2026-10-21\2017\grenoble_lts_2017_SIMPLE.pkl
[2018] loaded results\lts_networks\grenoble\fresh_2026-10-21\2018\grenoble_lts_2018_SIMPLE.pkl


[2019] loaded results\lts_networks\grenoble\fresh_2026-10-21\2019\grenoble_lts_2019_SIMPLE.pkl
[2020] loaded results\lts_networks\grenoble\fresh_2026-10-21\2020\grenoble_lts_2020_SIMPLE.pkl
[2021] loaded results\lts_networks\grenoble\fresh_2026-10-21\2021\grenoble_lts_2021_SIMPLE.pkl


[2022] loaded results\lts_networks\grenoble\fresh_2026-10-21\2022\grenoble_lts_2022_SIMPLE.pkl
[2026] loaded results\lts_networks\grenoble\fresh_2026-10-21\2026\grenoble_lts_2026_SIMPLE.pkl
Failed: none


## 3. Infrastructure metrics (LTS 5 excluded)

* `cycling_length_km` — total length of the LTS 1–4 edges;
* `lcc_ltsT_km` — length of the largest connected piece of network using only edges with LTS ≤ T;
* `normalized_network_length` (**N**) and `weighted_lcc_normalized` (**C**) divide by the same reference length
  for every year (the largest yearly cycling length of the city), so the values can be compared over time.

In [4]:
if failed:
    raise RuntimeError(f"Rerun section 2 first; missing: {sorted(failed)}")

print("LTS weights (kappa=%.1f):" % KAPPA, {t: round(w, 4) for t, w in exponential_weights(KAPPA).items()})
cols = ["year", "cycling_length_km", "normalized_network_length", "lcc_lts1_km", "lcc_lts2_km",
        "lcc_lts3_km", "lcc_lts4_km", "weighted_lcc_normalized", "lts5_edges", "lts5_mandatory_edges"]
all_metrics = {}
for name, graphs in networks.items():
    city = config.get_city(name)
    metrics = annual_metrics(graphs, kappa=KAPPA)
    csv_path, txt_path = write_metrics_files(metrics, config.run_dir(name, RUN_NAME), city.slug, name,
                                             RUN_NAME, MATCH, KAPPA)
    all_metrics[name] = metrics
    print(f"\n{name}  ->  {csv_path.relative_to(ROOT)}")
    display(metrics[cols].round(4))

LTS weights (kappa=1.0): {1: 0.6439, 2: 0.2369, 3: 0.0871, 4: 0.0321}



Grenoble  ->  results\lts_networks\grenoble\fresh_2026-10-21\grenoble_2017_2022_2026_SIMPLIFIED_metrics.csv


,year,cycling_length_km,normalized_network_length,lcc_lts1_km,lcc_lts2_km,lcc_lts3_km,lcc_lts4_km,weighted_lcc_normalized,lts5_edges,lts5_mandatory_edges
0,2017,389.0346,0.9301,15.3533,335.2093,385.0795,389.0346,0.3235,0,0
1,2018,392.6995,0.9389,15.3789,338.0646,388.7444,392.6995,0.3262,0,0
2,2019,403.6980,0.9652,15.9680,346.7153,400.0826,403.6980,0.3352,0,0
3,2020,405.0395,0.9684,29.1247,351.4929,402.0220,405.0395,0.3587,0,0
4,2021,413.4887,0.9886,32.6927,361.2714,410.5041,413.4887,0.3722,0,0
5,2022,415.4487,0.9933,31.0489,368.6580,413.1105,415.4487,0.3745,0,0
6,2026,418.2624,1.0000,47.0575,381.9174,415.7503,418.2624,0.4074,81,5


## 4. The 2026 network (bikes + car roads)

The 2026 graph is the starting point of the optimisation (steps 4–6). LTS 5 roads flagged *must upgrade* are
dangerous car roads where cycling is allowed; *prohibited* roads (motorways, `bicycle=no`) can never be upgraded.

In [5]:
for name, graphs in networks.items():
    G = graphs[config.CURRENT_YEAR]
    lts5 = [d for _, _, d in G.edges(data=True) if d["key"] == 5]
    print(f"{name} {config.CURRENT_YEAR}: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
    print("  edges per LTS:", dict(sorted(check_graph(G, verbose=False)["lts_counts"].items())))
    print(f"  LTS 5 edges: {len(lts5):,} | must upgrade to LTS 1: {sum(d['mandatory_upgrade_to_lts1'] for d in lts5):,}"
          f" | cycling prohibited: {sum(bool(d.get('bicycle_prohibited')) for d in lts5):,}")
    print(f"  LTS 5 length: {sum(d['distance'] for d in lts5)/1000:.2f} km, of which must-upgrade "
          f"{sum(d['distance'] for d in lts5 if d['mandatory_upgrade_to_lts1'])/1000:.2f} km")

Grenoble 2026: 5,664 nodes, 7,729 edges
  edges per LTS: {1: 3817, 2: 3344, 3: 465, 4: 22, 5: 81}
  LTS 5 edges: 81 | must upgrade to LTS 1: 5 | cycling prohibited: 69
  LTS 5 length: 32.59 km, of which must-upgrade 1.41 km


## 5. Maps

Simplified networks are drawn as straight lines between their remaining nodes (the "topology" the algorithms
work on). Green = LTS 1, light green = LTS 2, orange = LTS 3, red = LTS 4, grey = LTS 5, purple = LTS 5 to be
upgraded to LTS 1. The second map compares 2026 before and after simplification (needs the large intermediate
file `..._pre_simplification.pkl`, which is created by a fresh build and is not stored in git).

In [6]:
if SHOW_MAPS:
    for name, graphs in networks.items():
        for year in (min(graphs), config.CURRENT_YEAR):
            plot_topology(graphs[year], title=f"{name} {year} — simplified LTS network")

In [7]:
if SHOW_MAPS:
    for name, graphs in networks.items():
        slug, year = config.get_city(name).slug, config.CURRENT_YEAR
        before = config.run_dir(name, RUN_NAME) / str(year) / f"{slug}_lts5_{year}_pre_simplification.pkl"
        if before.exists():
            plot_before_after(config.load_pickle(before), graphs[year], title=f"{name} {year}")
        else:
            print(f"{name}: {before.name} not available (only created by a fresh build).")
        leftover_degree2(graphs[year])

Degree-2 nodes kept: 53 (all because the LTS changes there) | wrongly kept: 0
